In [ ]:
"""
Project 02: Predictive Maintenance
Step 02: Cost-Sensitive Model Training
"""

In [ ]:
import warnings

In [ ]:
warnings.filterwarnings("ignore")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pickle
import os
import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

In [ ]:
BASE = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RAW = os.path.join(BASE, "data", "raw")
PROCESSED = os.path.join(BASE, "data", "processed")
MODELS = os.path.join(BASE, "models")
CHARTS = os.path.join(BASE, "charts")

In [ ]:
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
os.makedirs(CHARTS, exist_ok=True)

In [ ]:
print("=" * 60)
print("STEP 02: Cost-Sensitive Model Training")
print("=" * 60)
start_time = time.time()

In [ ]:
# 1. Load Processed Data
print("\n[1/5] Loading processed data...")
df = pd.read_csv(os.path.join(PROCESSED, "processed_data.csv"))
X = df.drop(columns=["Target"])
y = df["Target"]
print(f"  Shape: {df.shape}")
print(f"  Features: {list(X.columns)}")

In [ ]:
# 2. Encode Target
print("\n[2/5] Encoding target labels...")
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"  Classes: {list(le.classes_)}")
print(f"  Encoded mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

In [ ]:
# 3. Train/Test Split
print("\n[3/5] Splitting data (80/20 stratified)...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"  Train: {X_train.shape[0]} samples")
print(f"  Test:  {X_test.shape[0]} samples")

In [ ]:
# Save encoder and test data
with open(os.path.join(MODELS, "label_encoder.pkl"), "wb") as f:
    pickle.dump(le, f)
with open(os.path.join(PROCESSED, "X_test.pkl"), "wb") as f:
    pickle.dump(X_test, f)
with open(os.path.join(PROCESSED, "y_test.pkl"), "wb") as f:
    pickle.dump(y_test, f)
print("  Saved: label_encoder.pkl, X_test.pkl, y_test.pkl")

In [ ]:
# 4. Compute Sample Weights for Cost-Sensitive Learning
print("\n[4/5] Computing balanced sample weights...")
print("  Applying Cost-Sensitive Learning to handle 96.5% class imbalance...")
print("  Missing a failure (False Negative) is heavily penalized.")
sample_weights = compute_sample_weight("balanced", y_train)
print(
    f"  Sample weights computed: min={sample_weights.min():.2f}, max={sample_weights.max():.2f}"
)

In [ ]:
# 5. Train Models
print("\n[5/5] Training cost-sensitive ensemble models...")
results = {}

In [ ]:
# Model 1: Decision Tree (Baseline - Interpretable Rules)
print("\n  --- Model 1: Decision Tree (Baseline) ---")
for i in tqdm(range(1), desc="  Training Decision Tree"):
    dt = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42)
    dt.fit(X_train, y_train)
dt_train_acc = dt.score(X_train, y_train)
dt_test_acc = dt.score(X_test, y_test)
print(f"  Train Acc: {dt_train_acc:.4f} | Test Acc: {dt_test_acc:.4f}")
with open(os.path.join(MODELS, "decision_tree.pkl"), "wb") as f:
    pickle.dump(dt, f)
results["Decision Tree"] = {"train_acc": dt_train_acc, "test_acc": dt_test_acc}

In [ ]:
# Model 2: Random Forest (Cost-Sensitive Bagging)
print("\n  --- Model 2: Random Forest (Bagging) ---")
for i in tqdm(range(1), desc="  Training Random Forest"):
    rf = RandomForestClassifier(
        n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=42
    )
    rf.fit(X_train, y_train)
rf_train_acc = rf.score(X_train, y_train)
rf_test_acc = rf.score(X_test, y_test)
print(f"  Train Acc: {rf_train_acc:.4f} | Test Acc: {rf_test_acc:.4f}")
with open(os.path.join(MODELS, "random_forest.pkl"), "wb") as f:
    pickle.dump(rf, f)
results["Random Forest"] = {"train_acc": rf_train_acc, "test_acc": rf_test_acc}

In [ ]:
# Feature importance from Random Forest
print("\n  Feature Importances (Random Forest):")
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(
    ascending=False
)
for feat, imp in feat_imp.items():
    print(f"    {feat}: {imp:.4f}")

In [ ]:
# Model 3: XGBoost (Cost-Sensitive Boosting with GPU fallback)
print("\n  --- Model 3: XGBoost (Boosting) ---")
for i in tqdm(range(1), desc="  Training XGBoost"):
    try:
        xgb = XGBClassifier(tree_method="hist", device="cuda", random_state=42)
        xgb.fit(X_train, y_train, sample_weight=sample_weights)
        print("  GPU acceleration: ENABLED")
    except Exception as e:
        print(f"  GPU not available, falling back to CPU: {e}")
        xgb = XGBClassifier(
            tree_method="hist", device="cpu", n_jobs=-1, random_state=42
        )
        xgb.fit(X_train, y_train, sample_weight=sample_weights)

In [ ]:
xgb_train_acc = xgb.score(X_train, y_train)
xgb_test_acc = xgb.score(X_test, y_test)
print(f"  Train Acc: {xgb_train_acc:.4f} | Test Acc: {xgb_test_acc:.4f}")
with open(os.path.join(MODELS, "xgboost.pkl"), "wb") as f:
    pickle.dump(xgb, f)
results["XGBoost"] = {"train_acc": xgb_train_acc, "test_acc": xgb_test_acc}

In [ ]:
# 6. Save Training Summary
print("\n  Training Summary:")
print(f"  {'Model':<20} {'Train Acc':>10} {'Test Acc':>10}")
print(f"  {'-' * 40}")
for model_name, accs in results.items():
    print(f"  {model_name:<20} {accs['train_acc']:>10.4f} {accs['test_acc']:>10.4f}")

In [ ]:
# Save results
import json

In [ ]:
with open(os.path.join(MODELS, "training_results.json"), "w") as f:
    json.dump(results, f, indent=4)

In [ ]:
# 7. Generate Training Charts
print("\n  Generating training charts...")

In [ ]:
# Chart: Feature Importance
for i in tqdm(range(1), desc="  Feature importance chart"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # RF importance
    feat_imp.plot(kind="barh", ax=axes[0], color="#3498db", edgecolor="black")
    axes[0].set_title("Random Forest Feature Importance", fontsize=12)
    axes[0].set_xlabel("Importance")
    axes[0].invert_yaxis()
    # XGB importance
    xgb_feat_imp = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(
        ascending=True
    )
    xgb_feat_imp.plot(kind="barh", ax=axes[1], color="#e74c3c", edgecolor="black")
    axes[1].set_title("XGBoost Feature Importance", fontsize=12)
    axes[1].set_xlabel("Importance")
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "feature_importance.png"), dpi=150)
    plt.close()

In [ ]:
# Chart: Model Comparison
for i in tqdm(range(1), desc="  Model comparison chart"):
    fig, ax = plt.subplots(figsize=(8, 5))
    model_names = list(results.keys())
    train_accs = [results[m]["train_acc"] for m in model_names]
    test_accs = [results[m]["test_acc"] for m in model_names]
    x_pos = np.arange(len(model_names))
    width = 0.35
    bars1 = ax.bar(
        x_pos - width / 2,
        train_accs,
        width,
        label="Train",
        color="#3498db",
        edgecolor="black",
    )
    bars2 = ax.bar(
        x_pos + width / 2,
        test_accs,
        width,
        label="Test",
        color="#e74c3c",
        edgecolor="black",
    )
    ax.set_ylabel("Accuracy")
    ax.set_title("Model Training vs Test Accuracy", fontsize=14)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_names, rotation=15)
    ax.legend()
    ax.set_ylim(0.9, 1.005)
    for bar in bars1:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.4f}",
            ha="center",
            fontsize=8,
        )
    for bar in bars2:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.001,
            f"{bar.get_height():.4f}",
            ha="center",
            fontsize=8,
        )
    plt.tight_layout()
    plt.savefig(os.path.join(CHARTS, "training_comparison.png"), dpi=150)
    plt.close()

In [ ]:
elapsed = time.time() - start_time
print(f"\n{'=' * 60}")
print(f"STEP 02 COMPLETE | Time: {elapsed:.1f}s")
print(f"{'=' * 60}")